# Basic Cross-Validation Example

This notebook demonstrates the basic workflow for running cross-validation with the Neural-Forecast system.

## What you'll learn:
- How to prepare data for CV
- How to load and configure models
- How to run cross-validation
- How to interpret basic results

## 1. Setup and Imports

In [ ]:
# Standard imports
import pandas as pd
import numpy as np
import yaml
from pathlib import Path
import warnings
warnings.filterwarnings('ignore')

# Add project root to path
import sys
sys.path.append('../..')

# Neural-Forecast imports
from neuralforecast import NeuralForecast
from neuralforecast.models import NHITS, TiDE
from neuralforecast.losses.pytorch import DistributionLoss

# Project imports
from cv.runner import run_cv, summarize_cv
from utils.validate import assert_regular_grid, assert_utc_eob

print("Imports successful!")

## 2. Create Sample Data

For this example, we'll create synthetic BTC-like data. In production, you would load real data from `data/processed/`.

In [ ]:
# Create synthetic BTC-like data
np.random.seed(42)

# Generate timestamps
n_periods = 4000  # About 40 days of 15-minute data
dates = pd.date_range(
    start='2024-01-01', 
    periods=n_periods, 
    freq='15min',
    tz='UTC'
)

# Generate synthetic returns with volatility clustering
returns = []
volatility = 0.001
for i in range(n_periods):
    # Volatility clustering
    volatility = 0.9 * volatility + 0.1 * np.abs(np.random.randn()) * 0.002
    returns.append(np.random.randn() * volatility)

# Create DataFrame
df = pd.DataFrame({
    'unique_id': 'BTC',
    'ds': dates,
    'y': returns
})

# Add some simple features
df['returns_ma_4'] = df['y'].rolling(4).mean().shift(1)
df['returns_std_16'] = df['y'].rolling(16).std().shift(1)
df['returns_abs'] = df['y'].abs().shift(1)

# Fill NaN values from rolling
df = df.fillna(0)

print(f"Created dataset with shape: {df.shape}")
print(f"Date range: {df['ds'].min()} to {df['ds'].max()}")
print(f"\nFirst few rows:")
df.head()

## 3. Validate Data Quality

Always validate your data before running CV to catch issues early.

In [ ]:
# Run validation checks
try:
    assert_regular_grid(df, "15min")
    print("✓ Data has regular 15-minute grid")
except AssertionError as e:
    print(f"✗ Grid validation failed: {e}")

try:
    assert_utc_eob(df, "15min")
    print("✓ Timestamps are UTC end-of-bar")
except AssertionError as e:
    print(f"✗ Timestamp validation failed: {e}")

# Check for NaN values
nan_counts = df.isna().sum()
if nan_counts.sum() == 0:
    print("✓ No NaN values found")
else:
    print(f"✗ NaN values found:")
    print(nan_counts[nan_counts > 0])

# Basic statistics
print(f"\nTarget statistics:")
print(f"  Mean: {df['y'].mean():.6f}")
print(f"  Std:  {df['y'].std():.6f}")
print(f"  Min:  {df['y'].min():.6f}")
print(f"  Max:  {df['y'].max():.6f}")

## 4. Configure Cross-Validation

We'll use a simplified configuration for this example.

In [ ]:
# Create configuration
cfg = {
    'seed': 42,
    'freq': '15min',
    'h': 4,  # 1-hour horizon
    
    # CV parameters (reduced for example)
    'n_windows': 3,  # Use only 3 windows for quick demo
    'step_size': 4,  # Step by horizon
    'val_size': 32,  # 2 hours validation
    'refit': True,   # Refit each window
    
    # Feature lists
    'hist_exog_list': ['returns_ma_4', 'returns_std_16', 'returns_abs'],
    'futr_exog_list': [],
    'stat_exog_list': []
}

print("Configuration:")
print(f"  Horizon: {cfg['h']} steps ({cfg['h'] * 15} minutes)")
print(f"  CV windows: {cfg['n_windows']}")
print(f"  Validation size: {cfg['val_size']} steps ({cfg['val_size'] * 15} minutes)")
print(f"  Historical features: {cfg['hist_exog_list']}")

## 5. Create Models

We'll create two simple models for demonstration.

In [ ]:
# Create model instances
models = [
    NHITS(
        h=cfg['h'],
        input_size=96,  # 24 hours of history
        loss=DistributionLoss('StudentT'),
        learning_rate=0.001,
        max_steps=500,  # Reduced for demo
        batch_size=32,
        hist_exog_list=cfg['hist_exog_list'],
        random_seed=cfg['seed'],
        alias='NHITS_demo'
    ),
    
    TiDE(
        h=cfg['h'],
        input_size=96,
        loss=DistributionLoss('StudentT'),
        learning_rate=0.001,
        max_steps=500,  # Reduced for demo
        batch_size=32,
        hist_exog_list=cfg['hist_exog_list'],
        random_seed=cfg['seed'],
        alias='TiDE_demo'
    )
]

print(f"Created {len(models)} models:")
for model in models:
    print(f"  - {model.alias}")

## 6. Initialize NeuralForecast

In [ ]:
# Create NeuralForecast instance
nf = NeuralForecast(
    models=models,
    freq=cfg['freq']
)

print("NeuralForecast instance created")
print(f"Number of models: {len(nf.models)}")
print(f"Frequency: {nf.freq}")

## 7. Fit Models

Models must be fitted before running cross-validation.

In [ ]:
# Fit the models
print("Fitting models...")
print("This may take a few minutes...")

nf.fit(
    df=df,
    val_size=cfg['val_size']
)

print("\n✓ Models fitted successfully!")

## 8. Run Cross-Validation

Now we run the actual cross-validation with proper windowing.

In [ ]:
# Run cross-validation
print(f"Running cross-validation with {cfg['n_windows']} windows...")
print("This may take several minutes...\n")

cv_results = run_cv(
    nf=nf,
    df=df,
    cfg=cfg,
    use_conformal=False  # Disable conformal for basic example
)

print(f"\n✓ Cross-validation completed!")
print(f"Results shape: {cv_results.shape}")
print(f"\nColumns in results:")
print(cv_results.columns.tolist())

## 9. Examine Raw CV Results

In [ ]:
# Look at the structure of CV results
print("First few rows of CV results:")
display(cv_results.head())

# Check unique cutoff dates (CV windows)
cutoffs = cv_results['cutoff'].unique()
print(f"\nNumber of CV windows: {len(cutoffs)}")
print("Cutoff dates:")
for i, cutoff in enumerate(cutoffs, 1):
    print(f"  Window {i}: {cutoff}")

# Check predictions for each model
print("\nModel predictions available:")
for col in cv_results.columns:
    if col.startswith(('NHITS', 'TiDE')):
        null_pct = cv_results[col].isna().mean() * 100
        print(f"  {col}: {100-null_pct:.1f}% complete")

## 10. Compute Metrics and Generate Summary

In [ ]:
# Get model names
model_names = [model.alias for model in models]

# Generate comprehensive summary
print("Computing metrics...")
summary = summarize_cv(
    cv_df=cv_results,
    model_names=model_names
)

print("\n✓ Metrics computed successfully!")
print(f"\nSummary contains:")
for key in summary.keys():
    print(f"  - {key}")

## 11. View Leaderboard

In [ ]:
# Display model leaderboard
print("=" * 60)
print("MODEL LEADERBOARD")
print("=" * 60)
print("\nModels ranked by sCRPS (lower is better):")
display(summary['leaderboard'])

# Identify best model
best_model = summary['leaderboard'].index[0]
best_scrps = summary['leaderboard'].loc[best_model, 'sCRPS_mean']
print(f"\n🏆 Best model: {best_model}")
print(f"   sCRPS: {best_scrps:.4f}")

## 12. Analyze Coverage

In [ ]:
# Check coverage metrics
print("=" * 60)
print("COVERAGE ANALYSIS")
print("=" * 60)

for model_name in model_names:
    if model_name in summary['coverage']:
        print(f"\n{model_name}:")
        cov = summary['coverage'][model_name]
        
        for level in [80, 90, 95]:
            actual = cov.get(str(level), 0) * 100
            target_min = level - 2
            target_max = level + 2
            
            # Check if within tolerance
            if target_min <= actual <= target_max:
                status = "✓"
            else:
                status = "✗"
            
            print(f"  {level}% level: {actual:.1f}% {status} (target: {level}±2%)")

## 13. Visualize Results

Let's create some simple visualizations to better understand the results.

In [ ]:
import matplotlib.pyplot as plt

# Plot predictions vs actuals for one CV window
fig, axes = plt.subplots(len(model_names), 1, figsize=(12, 4*len(model_names)))
if len(model_names) == 1:
    axes = [axes]

# Use first CV window
first_cutoff = cutoffs[0]
window_data = cv_results[cv_results['cutoff'] == first_cutoff].copy()

for idx, model_name in enumerate(model_names):
    ax = axes[idx]
    
    # Plot actual values
    ax.plot(window_data['ds'], window_data['y'], 
            'k-', label='Actual', alpha=0.7)
    
    # Plot predictions
    pred_col = model_name
    if pred_col in window_data.columns:
        ax.plot(window_data['ds'], window_data[pred_col], 
                'b-', label='Prediction', alpha=0.7)
    
    # Plot prediction intervals if available
    lo_col = f'{model_name}-lo-90'
    hi_col = f'{model_name}-hi-90'
    if lo_col in window_data.columns and hi_col in window_data.columns:
        ax.fill_between(window_data['ds'], 
                        window_data[lo_col], 
                        window_data[hi_col],
                        alpha=0.2, label='90% PI')
    
    ax.set_title(f'{model_name} - Window 1')
    ax.set_xlabel('Time')
    ax.set_ylabel('Returns')
    ax.legend(loc='upper right')
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 14. Compare Model Performance

In [ ]:
# Create comparison bar chart
metrics_to_plot = ['sCRPS_mean', 'MAE', 'RMSE']
fig, axes = plt.subplots(1, len(metrics_to_plot), figsize=(12, 4))

for idx, metric in enumerate(metrics_to_plot):
    ax = axes[idx]
    values = summary['leaderboard'][metric].values
    models = summary['leaderboard'].index
    
    bars = ax.bar(range(len(models)), values)
    ax.set_xticks(range(len(models)))
    ax.set_xticklabels(models, rotation=45)
    ax.set_title(metric)
    ax.set_ylabel('Value')
    
    # Color best performer
    best_idx = np.argmin(values)
    bars[best_idx].set_color('green')

plt.suptitle('Model Performance Comparison', fontsize=14)
plt.tight_layout()
plt.show()

## 15. Save Results

Finally, let's save the CV results for later analysis.

In [ ]:
# Create output directory
output_dir = Path('../../experiments/demo')
output_dir.mkdir(parents=True, exist_ok=True)

# Save CV predictions
cv_file = output_dir / 'cv_predictions.parquet'
cv_results.to_parquet(cv_file)
print(f"✓ CV predictions saved to: {cv_file}")

# Save metrics summary
import json
metrics_file = output_dir / 'metrics_summary.json'
with open(metrics_file, 'w') as f:
    # Convert DataFrames to dictionaries for JSON
    summary_json = {
        'leaderboard': summary['leaderboard'].to_dict(),
        'metrics': summary['metrics'],
        'coverage': summary['coverage'],
        'best_models': summary.get('best_models', {})
    }
    json.dump(summary_json, f, indent=2, default=str)
print(f"✓ Metrics summary saved to: {metrics_file}")

# Save leaderboard as CSV
leaderboard_file = output_dir / 'leaderboard.csv'
summary['leaderboard'].to_csv(leaderboard_file)
print(f"✓ Leaderboard saved to: {leaderboard_file}")

## Summary

In this notebook, we covered:

1. ✅ Data preparation and validation
2. ✅ Model configuration and initialization
3. ✅ Running NeuralForecast cross-validation
4. ✅ Computing metrics (sCRPS, MAE, RMSE, coverage)
5. ✅ Creating model leaderboard
6. ✅ Analyzing prediction interval coverage
7. ✅ Visualizing predictions
8. ✅ Saving results for later use

### Key Takeaways:

- **sCRPS** is our primary metric for ranking models (lower is better)
- **Coverage** should be within ±2% of nominal levels for well-calibrated models
- **Cross-validation** respects temporal ordering with expanding windows
- Results are automatically saved for reproducibility

### Next Steps:

- See `02_metrics_analysis.ipynb` for deeper metrics understanding
- See `03_calibration_diag.ipynb` for PIT analysis and calibration
- See `04_model_selection.ipynb` for advanced selection strategies
- See `05_performance_tuning.ipynb` for optimization tips